# SimpleTM — Exploratory Data Analysis
## BTP Project: SWT vs FFT Tokenization for Multivariate Time Series Forecasting

This notebook explores the ETTh1 dataset and analyzes signal characteristics relevant to comparing:
- **SWT (Stationary Wavelet Transform)** tokenization
- **FFT (Fast Fourier Transform)** tokenization
- **Hybrid (SWT + Geometric Attention)** — the original SimpleTM

### Sections
1. Setup & Data Loading
2. Basic Statistics & Overview
3. Time Series Visualization
4. Correlation & Cross-Variate Analysis
5. Stationarity Testing (ADF)
6. Trend & Seasonality Decomposition
7. Frequency Spectrum Analysis (FFT)
8. Wavelet Decomposition Analysis (SWT)
9. SWT vs FFT Representation Comparison
10. Train/Val/Test Split Visualization
11. Model Results Comparison

## 1. Setup & Data Loading

In [ ]:
!pip install PyWavelets gdown statsmodels --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import pywt
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 12
sns.set_palette('husl')
print('Libraries loaded.')

In [ ]:
import os, gdown, zipfile

FILE_ID = '1hTpUrhe1yEIGa9mCiGxM5rDyzlYKAnyx'
OUTPUT = '/kaggle/working/dataset.zip'
DATASET_DIR = '/kaggle/working/dataset'

if not os.path.exists(DATASET_DIR):
    gdown.download(f'https://drive.google.com/uc?id={FILE_ID}', OUTPUT, quiet=False)
    os.makedirs(DATASET_DIR, exist_ok=True)
    with zipfile.ZipFile(OUTPUT, 'r') as z:
        z.extractall(DATASET_DIR)
    print('Dataset extracted.')

# Find ETTh1.csv
ett_path = None
for root, dirs, files in os.walk(DATASET_DIR):
    for f in files:
        if f == 'ETTh1.csv':
            ett_path = os.path.join(root, f)
            break
print(f'ETTh1 path: {ett_path}')

df = pd.read_csv(ett_path)
df['date'] = pd.to_datetime(df['date'])
df.set_index('date', inplace=True)
print(f'Shape: {df.shape}')
df.head()

## 2. Basic Statistics & Overview

In [ ]:
print('='*60)
print('DATASET OVERVIEW')
print('='*60)
print(f'Time range : {df.index[0]} → {df.index[-1]}')
print(f'Duration   : {(df.index[-1] - df.index[0]).days} days')
print(f'Samples    : {len(df)}')
print(f'Features   : {list(df.columns)}')
print(f'Freq       : Hourly (inferred: {pd.infer_freq(df.index[:100])})')
print(f'\nMissing values:\n{df.isnull().sum()}')
print(f'\n{df.describe().round(3)}')

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for i, col in enumerate(df.columns):
    ax = axes[i//4, i%4]
    df[col].hist(bins=50, ax=ax, alpha=0.7, edgecolor='black')
    ax.set_title(col, fontweight='bold')
    ax.axvline(df[col].mean(), color='red', linestyle='--', label=f'μ={df[col].mean():.1f}')
    ax.legend(fontsize=8)
if len(df.columns) < 8:
    axes[1, 3].axis('off')
fig.suptitle('Feature Distributions', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Time Series Visualization

In [ ]:
fig, axes = plt.subplots(len(df.columns), 1, figsize=(16, 3*len(df.columns)), sharex=True)
colors = plt.cm.tab10(np.linspace(0, 1, len(df.columns)))
for i, col in enumerate(df.columns):
    axes[i].plot(df.index, df[col], color=colors[i], linewidth=0.5, alpha=0.8)
    axes[i].set_ylabel(col, fontweight='bold')
    axes[i].fill_between(df.index, df[col].min(), df[col], alpha=0.1, color=colors[i])
fig.suptitle('ETTh1 — All Variates Over Time', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Zoom into first 2 weeks (336 hours)
zoom = df.iloc[:336]
fig, ax = plt.subplots(figsize=(16, 5))
for col in df.columns:
    ax.plot(zoom.index, zoom[col], label=col, linewidth=1.5)
ax.set_title('First 2 Weeks — All Variates (Zoomed In)', fontsize=14, fontweight='bold')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
ax.set_xlabel('Time')
plt.tight_layout()
plt.show()

## 4. Correlation & Cross-Variate Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Pearson correlation
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            ax=axes[0], square=True, vmin=-1, vmax=1)
axes[0].set_title('Pearson Correlation', fontweight='bold')

# Spearman (rank) correlation
corr_s = df.corr(method='spearman')
sns.heatmap(corr_s, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            ax=axes[1], square=True, vmin=-1, vmax=1)
axes[1].set_title('Spearman Rank Correlation', fontweight='bold')
plt.tight_layout()
plt.show()

print('\nHighly correlated pairs (|r| > 0.7):')
for i in range(len(corr.columns)):
    for j in range(i+1, len(corr.columns)):
        if abs(corr.iloc[i,j]) > 0.7:
            print(f'  {corr.columns[i]} ↔ {corr.columns[j]}: r = {corr.iloc[i,j]:.3f}')

In [ ]:
# Rolling correlation of each feature with target (OT)
fig, ax = plt.subplots(figsize=(16, 5))
window = 24*7  # 1 week
for col in df.columns[:-1]:  # Exclude OT itself
    rolling_corr = df[col].rolling(window).corr(df['OT'])
    ax.plot(df.index, rolling_corr, label=col, alpha=0.7, linewidth=0.8)
ax.axhline(0, color='black', linestyle='--', alpha=0.5)
ax.set_title(f'Rolling Correlation with Target (OT) — Window={window}h', fontweight='bold')
ax.legend(bbox_to_anchor=(1.02, 1))
ax.set_ylim(-1, 1)
plt.tight_layout()
plt.show()

## 5. Stationarity Testing (ADF)

In [ ]:
from statsmodels.tsa.stattools import adfuller

print(f'{"Feature":<10} {"ADF Stat":>10} {"p-value":>10} {"Stationary?":>12}')
print('-'*45)
for col in df.columns:
    result = adfuller(df[col].dropna(), maxlag=48)
    stationary = '✓ Yes' if result[1] < 0.05 else '✗ No'
    print(f'{col:<10} {result[0]:>10.4f} {result[1]:>10.6f} {stationary:>12}')

print('\n→ Non-stationary features benefit more from SWT/FFT decomposition')
print('  as it separates trend from oscillatory components.')

## 6. Trend & Seasonality Decomposition

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

# Decompose target variable OT
decomp = seasonal_decompose(df['OT'].iloc[:24*30*4], model='additive', period=24)

fig, axes = plt.subplots(4, 1, figsize=(16, 12), sharex=True)
decomp.observed.plot(ax=axes[0], title='Original (OT)', linewidth=0.8)
decomp.trend.plot(ax=axes[1], title='Trend', linewidth=1.2, color='orange')
decomp.seasonal.plot(ax=axes[2], title='Seasonal (24h period)', linewidth=0.8, color='green')
decomp.resid.plot(ax=axes[3], title='Residual', linewidth=0.5, color='red', alpha=0.7)
for ax in axes:
    ax.title.set_fontweight('bold')
fig.suptitle('Seasonal Decomposition of OT (Period=24h)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print(f'Trend variance     : {decomp.trend.dropna().var():.4f}')
print(f'Seasonal variance  : {decomp.seasonal.var():.4f}')
print(f'Residual variance  : {decomp.resid.dropna().var():.4f}')
print(f'Seasonal strength  : {1 - decomp.resid.dropna().var() / (decomp.seasonal + decomp.resid).dropna().var():.4f}')

In [ ]:
# Autocorrelation for all features
from statsmodels.graphics.tsaplots import plot_acf

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for i, col in enumerate(df.columns):
    ax = axes[i//4, i%4]
    plot_acf(df[col].dropna(), lags=96, ax=ax, alpha=0.05, title=col)
    ax.set_xlabel('Lag (hours)')
if len(df.columns) < 8:
    axes[1, 3].axis('off')
fig.suptitle('Autocorrelation (96 lags = 4 days)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 7. Frequency Spectrum Analysis (FFT)

This section directly relates to our **FFT tokenization** variant — it shows what frequency components exist in the data and how FFT band splitting captures them.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for i, col in enumerate(df.columns):
    ax = axes[i//4, i%4]
    signal = df[col].values - df[col].mean()
    fft_vals = np.fft.rfft(signal)
    freqs = np.fft.rfftfreq(len(signal), d=1)  # 1 hour
    power = np.abs(fft_vals)**2
    # Skip DC component
    ax.semilogy(freqs[1:200], power[1:200], linewidth=0.8)
    ax.set_title(col, fontweight='bold')
    ax.set_xlabel('Freq (cycles/hour)')
    ax.set_ylabel('Power')
    # Mark dominant frequencies
    top_idx = np.argsort(power[1:200])[-3:] + 1
    for idx in top_idx:
        period = 1/freqs[idx] if freqs[idx] > 0 else np.inf
        ax.axvline(freqs[idx], color='red', alpha=0.3, linestyle='--')
        ax.annotate(f'{period:.0f}h', (freqs[idx], power[idx]), fontsize=7, color='red')
if len(df.columns) < 8:
    axes[1, 3].axis('off')
fig.suptitle('Power Spectrum (FFT) — Top 3 Dominant Periods Marked', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# FFT Band Decomposition (as our FFT model does it)
signal = df['OT'].values[:96] - df['OT'].values[:96].mean()  # One input window
m = 3  # Same as model config
fft_vals = np.fft.rfft(signal)
n_freqs = len(fft_vals)
band_size = max(1, n_freqs // (m + 1))

fig, axes = plt.subplots(m+2, 1, figsize=(14, 3*(m+2)), sharex=True)
axes[0].plot(signal, 'k-', linewidth=1.5, label='Original')
axes[0].set_title('Original Signal (OT, first 96 steps)', fontweight='bold')
axes[0].legend()

reconstructed = np.zeros_like(signal)
band_labels = ['Low Freq (Trend)', 'Low-Mid Freq', 'Mid-High Freq', 'High Freq (Detail)']
colors = ['#e74c3c', '#f39c12', '#2ecc71', '#3498db']
for i in range(m + 1):
    mask = np.zeros(n_freqs)
    start = i * band_size
    end = min((i+1)*band_size, n_freqs) if i < m else n_freqs
    mask[start:end] = 1.0
    band = np.fft.irfft(fft_vals * mask, n=len(signal))
    reconstructed += band
    lbl = band_labels[i] if i < len(band_labels) else f'Band {i}'
    axes[i+1].plot(band, color=colors[i%len(colors)], linewidth=1.2)
    axes[i+1].set_title(f'FFT Band {i}: {lbl} (freqs {start}-{end-1})', fontweight='bold')
    axes[i+1].fill_between(range(len(band)), 0, band, alpha=0.2, color=colors[i%len(colors)])

fig.suptitle(f'FFT Band Decomposition (m={m}, {m+1} bands) — As Used by SimpleTM_FFT',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

recon_error = np.mean((signal - reconstructed)**2)
print(f'Reconstruction MSE: {recon_error:.2e} (should be ~0 for perfect reconstruction)')

## 8. Wavelet Decomposition Analysis (SWT)

This section shows what the **SWT tokenization** (original SimpleTM) captures — multi-scale wavelet coefficients at different resolution levels.

In [ ]:
# SWT decomposition (as our SWT model does it)
signal_ot = df['OT'].values[:96].astype(np.float64)
signal_ot = signal_ot - signal_ot.mean()
wavelet = 'db1'  # Same as model default
m = 3
coeffs = pywt.swt(signal_ot, wavelet, level=m)

fig, axes = plt.subplots(m+2, 1, figsize=(14, 3*(m+2)), sharex=True)
axes[0].plot(signal_ot, 'k-', linewidth=1.5)
axes[0].set_title('Original Signal (OT, first 96 steps)', fontweight='bold')

swt_labels = ['Approx (A3) — Lowest Freq', 'Detail (D3) — Low Freq',
              'Detail (D2) — Mid Freq', 'Detail (D1) — High Freq']
colors = ['#e74c3c', '#9b59b6', '#2ecc71', '#3498db']

# Approximation coefficients (lowest frequency)
axes[1].plot(coeffs[-1][0], color=colors[0], linewidth=1.2)
axes[1].set_title(f'SWT Level {m}: {swt_labels[0]}', fontweight='bold')
axes[1].fill_between(range(len(coeffs[-1][0])), 0, coeffs[-1][0], alpha=0.2, color=colors[0])

# Detail coefficients (high to low frequency)
for i in range(m):
    detail = coeffs[m-1-i][1]  # Reversed order
    axes[i+2].plot(detail, color=colors[(i+1)%len(colors)], linewidth=1.2)
    lbl = swt_labels[i+1] if i+1 < len(swt_labels) else f'Detail D{m-i}'
    axes[i+2].set_title(f'SWT Level {m-i}: {lbl}', fontweight='bold')
    axes[i+2].fill_between(range(len(detail)), 0, detail, alpha=0.2, color=colors[(i+1)%len(colors)])

fig.suptitle(f'SWT Decomposition (wavelet={wavelet}, levels={m}) — As Used by SimpleTM/SimpleTM_SWT',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 9. SWT vs FFT Representation Comparison

Side-by-side comparison of how SWT and FFT decompose the same signal — the core question of our BTP project.

In [ ]:
signal = df['OT'].values[:96].astype(np.float64)
signal = signal - signal.mean()
m = 3

# FFT bands
fft_vals = np.fft.rfft(signal)
n_freqs = len(fft_vals)
band_size = max(1, n_freqs // (m + 1))
fft_bands = []
for i in range(m + 1):
    mask = np.zeros(n_freqs)
    start = i * band_size
    end = min((i+1)*band_size, n_freqs) if i < m else n_freqs
    mask[start:end] = 1.0
    fft_bands.append(np.fft.irfft(fft_vals * mask, n=len(signal)))

# SWT coeffs
swt_coeffs = pywt.swt(signal, 'db1', level=m)
swt_bands = [swt_coeffs[-1][0]]  # Approx
for i in range(m):
    swt_bands.append(swt_coeffs[m-1-i][1])  # Details

# Plot side by side
fig, axes = plt.subplots(m+1, 2, figsize=(16, 3*(m+1)), sharex=True)
level_names = ['Low Freq / Approx', 'Low-Mid / Detail 3', 'Mid-High / Detail 2', 'High / Detail 1']

for i in range(m+1):
    # FFT
    axes[i, 0].plot(fft_bands[i], color='#3498db', linewidth=1.2)
    axes[i, 0].fill_between(range(96), 0, fft_bands[i], alpha=0.15, color='#3498db')
    axes[i, 0].set_ylabel(level_names[i] if i < len(level_names) else f'Band {i}', fontsize=9)
    if i == 0: axes[i, 0].set_title('FFT Bands', fontsize=14, fontweight='bold')
    # SWT
    axes[i, 1].plot(swt_bands[i], color='#e74c3c', linewidth=1.2)
    axes[i, 1].fill_between(range(len(swt_bands[i])), 0, swt_bands[i], alpha=0.15, color='#e74c3c')
    if i == 0: axes[i, 1].set_title('SWT Coefficients', fontsize=14, fontweight='bold')

fig.suptitle('FFT vs SWT Decomposition — Same Signal, Different Tokenizations',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

# Energy distribution comparison
fft_energy = [np.sum(b**2) for b in fft_bands]
swt_energy = [np.sum(b**2) for b in swt_bands]
total_fft = sum(fft_energy)
total_swt = sum(swt_energy)

print('\nEnergy Distribution Across Bands:')
print(f'{"Band":<20} {"FFT %":>10} {"SWT %":>10}')
print('-'*42)
for i in range(m+1):
    name = level_names[i] if i < len(level_names) else f'Band {i}'
    print(f'{name:<20} {100*fft_energy[i]/total_fft:>9.1f}% {100*swt_energy[i]/total_swt:>9.1f}%')

## 10. Train / Validation / Test Split Visualization

In [ ]:
# ETTh1 standard split
n = len(df)
train_end = 12 * 30 * 24       # 12 months
val_end = train_end + 4*30*24   # 4 months
test_end = val_end + 4*30*24    # 4 months

fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(df.index[:train_end], df['OT'].iloc[:train_end], color='#2ecc71', label=f'Train ({train_end} pts)', linewidth=0.6)
ax.plot(df.index[train_end:val_end], df['OT'].iloc[train_end:val_end], color='#f39c12', label=f'Val ({val_end-train_end} pts)', linewidth=0.6)
ax.plot(df.index[val_end:test_end], df['OT'].iloc[val_end:test_end], color='#e74c3c', label=f'Test ({test_end-val_end} pts)', linewidth=0.6)
ax.axvline(df.index[train_end], color='black', linestyle='--', alpha=0.5)
ax.axvline(df.index[val_end], color='black', linestyle='--', alpha=0.5)
ax.set_title('ETTh1 — Train / Validation / Test Split (OT)', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

# Distribution shift analysis
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, (name, start, end, color) in enumerate([
    ('Train', 0, train_end, '#2ecc71'),
    ('Validation', train_end, val_end, '#f39c12'),
    ('Test', val_end, test_end, '#e74c3c')
]):
    subset = df['OT'].iloc[start:end]
    subset.hist(bins=40, ax=axes[i], color=color, alpha=0.7, edgecolor='black')
    axes[i].set_title(f'{name} (μ={subset.mean():.2f}, σ={subset.std():.2f})', fontweight='bold')
    axes[i].axvline(subset.mean(), color='black', linestyle='--')
fig.suptitle('Distribution Shift Across Splits — OT', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 11. Model Results Comparison

Results from our Kaggle training run (ETTh1, seq=96, pred=96, d_model=32).

In [ ]:
# Results from our training run
results = pd.DataFrame({
    'Model': ['SimpleTM\n(SWT+GeomAttn)', 'SimpleTM_SWT\n(SWT+GeomAttn)', 'SimpleTM_FFT\n(FFT+GeomAttn)'],
    'Tokenization': ['SWT', 'SWT', 'FFT'],
    'MSE': [0.451259, 0.451259, 0.457091],
    'MAE': [0.446162, 0.446162, 0.449469],
    'Params': [12856, 12856, 12808]
})

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = ['#2ecc71', '#3498db', '#e74c3c']

# MSE comparison
bars = axes[0].bar(results['Model'], results['MSE'], color=colors, edgecolor='black', alpha=0.85)
axes[0].set_title('MSE ↓', fontsize=14, fontweight='bold')
axes[0].set_ylim(0.44, 0.46)
for bar, val in zip(bars, results['MSE']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0003,
                f'{val:.4f}', ha='center', fontweight='bold', fontsize=10)

# MAE comparison
bars = axes[1].bar(results['Model'], results['MAE'], color=colors, edgecolor='black', alpha=0.85)
axes[1].set_title('MAE ↓', fontsize=14, fontweight='bold')
axes[1].set_ylim(0.44, 0.455)
for bar, val in zip(bars, results['MAE']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0003,
                f'{val:.4f}', ha='center', fontweight='bold', fontsize=10)

# Params comparison
bars = axes[2].bar(results['Model'], results['Params'], color=colors, edgecolor='black', alpha=0.85)
axes[2].set_title('Parameters', fontsize=14, fontweight='bold')
for bar, val in zip(bars, results['Params']):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                f'{val:,}', ha='center', fontweight='bold', fontsize=10)

fig.suptitle('Model Comparison — ETTh1 (96→96)', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('\nKey Findings:')
print(f'  • FFT MSE overhead vs SWT: {(0.457091-0.451259)/0.451259*100:.2f}%')
print(f'  • FFT MAE overhead vs SWT: {(0.449469-0.446162)/0.446162*100:.2f}%')
print(f'  • FFT saves {12856-12808} parameters (no wavelet filters)')
print(f'  • Both tokenizations work with geometric attention — FFT is competitive')

## Summary

### Key Observations:
1. **ETTh1 has strong 24h seasonality** — both SWT and FFT can capture this
2. **Non-stationary features** like HUFL, HULL benefit from multi-scale decomposition
3. **Energy concentration differs**: SWT concentrates in approximation coefficients, FFT spreads across low-freq bands
4. **FFT tokenization is competitive** with SWT (only ~1.3% MSE gap) while being simpler
5. **Distribution shift** exists between train/val/test — normalization (RevIN) is critical

### Implications for BTP:
- SWT's advantage comes from **time-localized** multi-resolution — useful for non-stationary signals
- FFT provides **global frequency** separation — cleaner for periodic signals
- The geometric attention (wedge product) provides the main performance boost regardless of tokenization